### Importing the Required Libraries

In [ ]:
import sys
!{sys.executable} -m pip uninstall -y transformers pydantic safetensors numpy
!{sys.executable} -m pip install "numpy<2.0" "pydantic==2.12.5"
!{sys.executable} -m pip install langchain langchain-community langchain-huggingface transformers safetensors chromadb
import os
os._exit(0)

Found existing installation: transformers 5.4.0
Uninstalling transformers-5.4.0:
  Successfully uninstalled transformers-5.4.0
Found existing installation: pydantic 2.12.5
Uninstalling pydantic-2.12.5:
  Successfully uninstalled pydantic-2.12.5
Found existing installation: safetensors 0.7.0
Uninstalling safetensors-0.7.0:
  Successfully uninstalled safetensors-0.7.0
Found existing installation: numpy 2.4.3
Uninstalling numpy-2.4.3:
  Successfully uninstalled numpy-2.4.3
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.25 requires safetensor

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import pipeline
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate

### Reading The Document

In [ ]:
loader=TextLoader("companyPolicies.txt")
document=loader.load()

In [ ]:
document

[Document(metadata={'source': 'companyPolicies.txt'}, page_content="1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to 

### Chunking Of The Document

In [ ]:
splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=0)
document=splitter.split_documents(document)

In [ ]:
document

[Document(metadata={'source': 'companyPolicies.txt'}, page_content='1.\tCode of Conduct'),
 Document(metadata={'source': 'companyPolicies.txt'}, page_content="Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decis

### Creating Embeddings And Storing In Chroma DB

In [ ]:
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(document, embeddings, persist_directory='./chroma_embeddings')

<>:2: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_13122/2961958774.py:2: SyntaxWarning: invalid escape sequence '\c'
  docsearch=Chroma.from_documents(document,embeddings,persist_directory='.\chroma_embeddings')
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
docsearch.embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### Loading The LLM

In [ ]:
model=pipeline(model='unsloth/Llama-3.2-3B-Instruct',
               task='text-generation')

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

### Making The Prompt Template

In [ ]:
template="Only Give the answer with respect to {context} of the {question} do not try to make it at any cost if answer not given in {context} just say don't know the answer"

In [ ]:
from langchain_core.prompts import PromptTemplate
proTemp=PromptTemplate(template=template,input_variables=["context", "question"])

### Building a QA Chain

In [ ]:
rag_chain = (
    {"context": docsearch.as_retriever(), "question": RunnablePassthrough()}
    | proTemp
    | (lambda x: x.to_string())
    | model
)

### Testing The Agent

In [ ]:
rag_chain.invoke("Can I harass a person on Company's Mail ?")

[{'generated_text': 'Only Give the answer with respect to [Document(metadata={\'source\': \'companyPolicies.txt\'}, page_content="Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and the importance of adhering to principles that maintain security, productivity, and legal compliance.\\nAcceptable Use: Company-provided internet and email services are primarily meant for job-related tasks. Limited personal use is allowed during non-work hours, provided it doesn\'t interfere with work responsibilities.\\nSecurity: Safeguard your login credentials, avoiding the sharing of passwords. Exercise caution with email attachments and links from unknown sources. Promptly report any unusual online activity or potential security breaches.\\nConfidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer

In [ ]:
rag_chain.invoke("If an employee is seen smoking an electronic cigarette while using a company-issued mobile phone to send offensive jokes to a colleague, which specific policies are being violated, and what are the potential combined consequences?")

[{'generated_text': 'Only Give the answer with respect to [Document(metadata={\'source\': \'companyPolicies.txt\'}, page_content=\'Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. This policy is in place to ensure a safe and healthy environment for all employees, visitors, and the general public.\\nDesignated Smoking Areas: Smoking is only permitted in designated smoking areas, as marked by appropriate signage. These areas have been chosen to minimize exposure to secondhand smoke and to maintain the overall cleanliness of the premises.\\nSmoking Restrictions: Smoking inside company buildings, offices, meeting rooms, and other enclosed spaces is strictly prohibited. This includes electronic cigarettes and vaping devices.\\nCompliance with Applicable Laws: All employees and visitors must adhere to relevant federal, state, and local smoking laws and regulations.\\nDisposal of Smoking Materials: Prope

### For Fixing Git Hub Rendering error

In [8]:
import json

with open("CompanyPolicyAgentUsingLangchain.ipynb", "r") as f:
    nb = json.load(f)

# Print EXACT current structure so we know what we're dealing with
w = nb["metadata"]["widgets"]
print("=== CURRENT STRUCTURE ===")
print(json.dumps({
    "widgets_keys": list(w.keys()),
    "inner_keys_sample": list(w.get("application/vnd.jupyter.widget-state+json", {}).keys())[:3],
    "has_state": "state" in w.get("application/vnd.jupyter.widget-state+json", {})
}, indent=2))

=== CURRENT STRUCTURE ===
{
  "widgets_keys": [
    "application/vnd.jupyter.widget-state+json"
  ],
  "inner_keys_sample": [
    "02c5ed70b9264748bfbc9e308554398f",
    "03154e3884e741148de7ec0d52838ecf",
    "0a4190a2a8584c44bdcd85ba6fded7fa"
  ],
  "has_state": false
}


In [9]:
import json

# Read
with open("CompanyPolicyAgentUsingLangchain.ipynb", "r") as f:
    nb = json.load(f)

# Delete widgets entirely
del nb["metadata"]["widgets"]

# Force write to a NEW file first
with open("CompanyPolicyAgentUsingLangchain_fixed.ipynb", "w") as f:
    json.dump(nb, f, indent=1)

# Verify the NEW file
with open("CompanyPolicyAgentUsingLangchain_fixed.ipynb", "r") as f:
    nb2 = json.load(f)
    print("widgets gone?", "widgets" not in nb2["metadata"])
    print("metadata keys:", list(nb2["metadata"].keys()))

widgets gone? True
metadata keys: ['accelerator', 'colab', 'kernelspec', 'language_info']


In [10]:
import os
os.replace("CompanyPolicyAgentUsingLangchain_fixed.ipynb", 
           "CompanyPolicyAgentUsingLangchain.ipynb")
print("Replaced! Now push to GitHub.")

Replaced! Now push to GitHub.


In [11]:
import json

with open("CompanyPolicyAgentUsingLangchain.ipynb", "r") as f:
    nb = json.load(f)

# Print EXACT current structure so we know what we're dealing with
w = nb["metadata"]["widgets"]
print("=== CURRENT STRUCTURE ===")
print(json.dumps({
    "widgets_keys": list(w.keys()),
    "inner_keys_sample": list(w.get("application/vnd.jupyter.widget-state+json", {}).keys())[:3],
    "has_state": "state" in w.get("application/vnd.jupyter.widget-state+json", {})
}, indent=2))

=== CURRENT STRUCTURE ===
{
  "widgets_keys": [
    "application/vnd.jupyter.widget-state+json"
  ],
  "inner_keys_sample": [
    "02c5ed70b9264748bfbc9e308554398f",
    "03154e3884e741148de7ec0d52838ecf",
    "0a4190a2a8584c44bdcd85ba6fded7fa"
  ],
  "has_state": false
}
